# VoleykoçAI — Ödev 3: Unsloth ile LoRA fine-tune

Bu notebook **Google Colab'da** çalışır. Unsloth, Triton'a dayandığı için Apple Silicon'da çalışmıyor; bu yüzden eğitimi burada yapıyorum.

**Çalıştırmadan önce:** üstteki menüden `Runtime → Change runtime type → T4 GPU` seç.

Sonuç: `berkcangumusisik/voleykoc-qwen3-4b-lora` adresinde yayımlanan bir LoRA adaptörü.

| | |
|---|---|
| Temel model | `unsloth/Qwen3-4B-Instruct-2507` |
| Veri seti | `berkcangumusisik/voleykoc-antrenorluk-tr` (Ödev 1) |
| Yöntem | 4-bit QLoRA, r=16 |
| Donanım | Colab T4 (16 GB) |

## 1) Kurulum ve GPU kontrolü

In [1]:
%pip install -q unsloth
# Unsloth'un en güncel halini çekiyorum; Colab'ın önyüklü sürümü bazen eski kalıyor.
%pip install -q --upgrade --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git

^C
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import torch

assert torch.cuda.is_available(), (
    "GPU yok. Runtime -> Change runtime type -> T4 GPU seçip yeniden başlat."
)
gpu = torch.cuda.get_device_properties(0)
print(f"GPU: {gpu.name}")
print(f"Bellek: {gpu.total_memory / 1024**3:.1f} GB")

## 2) Modeli yükle

4-bit quantization ile yüklüyorum: 4B parametrelik model böylece T4'ün 16 GB'ına rahat sığıyor.

In [ ]:
from unsloth import FastLanguageModel

MAX_SEQ_LENGTH = 2048
BASE_MODEL = "unsloth/Qwen3-4B-Instruct-2507"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,        # None = donanıma göre otomatik seç
    load_in_4bit=True,
)
print("Model yüklendi.")

## 3) Eğitim öncesi cevaplar

Fine-tune'un işe yarayıp yaramadığını görebilmek için aynı soruları **önce** soruyorum. Sonuçları rapora koyacağım.

In [ ]:
TEST_SORULARI = [
    "5-1 rotasyon sistemi nasıl çalışır?",
    "Manşet pasında topu kollarımın neresiyle karşılamalıyım?",
    "Libero hangi kuralları uygulamak zorunda?",
    "14 yaş altı bir gruba haftalık antrenman planını nasıl kurarım?",
    "Sıçrama yüksekliğimi artırmak için hangi çalışmaları yapmalıyım?",
]

SYSTEM = (
    "Sen VoleykoçAI'sın: Türkçe konuşan bir voleybol antrenörlük asistanısın. "
    "Teknik, taktik, antrenman planlaması, kondisyon ve oyun kuralları "
    "konularında somut ve uygulanabilir cevaplar verirsin."
)


def sor(soru, max_new_tokens=256):
    mesajlar = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": soru},
    ]
    girdi = tokenizer.apply_chat_template(
        mesajlar, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")
    cikti = model.generate(
        input_ids=girdi,
        max_new_tokens=max_new_tokens,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )
    # Sadece modelin ürettiği kısmı döndür, girdiyi tekrar yazdırma.
    return tokenizer.decode(cikti[0][girdi.shape[1]:], skip_special_tokens=True)


FastLanguageModel.for_inference(model)

once = {}
for soru in TEST_SORULARI:
    once[soru] = sor(soru)
    print(f"\n### {soru}\n{once[soru]}")

## 4) LoRA adaptörünü ekle

4B parametrenin tamamını eğitmek yerine, dikkat ve MLP katmanlarına küçük düşük-ranklı matrisler ekliyorum. Eğitilen parametre oranı %1'in altında kalıyor — Colab'da bu yüzden mümkün.

In [ ]:
SEED = 1337

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,          # 0 = Unsloth'un optimize ettiği yol
    bias="none",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    use_gradient_checkpointing="unsloth",  # uzun bağlamda bellek tasarrufu
    random_state=SEED,
)

egitilen = sum(p.numel() for p in model.parameters() if p.requires_grad)
toplam = sum(p.numel() for p in model.parameters())
print(f"Eğitilen parametre: {egitilen:,} / {toplam:,} "
      f"({egitilen / toplam * 100:.2f}%)")

## 5) Veri setini yükle ve sohbet şablonuna çevir

Ödev 1'de ürettiğim veri seti Hugging Face'te. `conversations` alanını Qwen'in sohbet şablonuna sokup düz metne çeviriyorum.

In [ ]:
from datasets import load_dataset

DATASET = "berkcangumusisik/voleykoc-antrenorluk-tr"

ds = load_dataset(DATASET, split="train")
print(ds)
print("\nÖrnek satır:")
print(ds[0])

In [ ]:
def bicimlendir(ornek):
    """magibu şemasındaki satırı Qwen sohbet şablonuna çevir."""
    mesajlar = [{"role": "system", "content": ornek["system"]}]
    mesajlar += [
        {"role": m["role"], "content": m["content"]}
        for m in ornek["conversations"]
    ]
    # add_generation_prompt=False: asistanın cevabı zaten veride var,
    # modelin onu üretmesini istiyoruz.
    return {
        "text": tokenizer.apply_chat_template(
            mesajlar, tokenize=False, add_generation_prompt=False
        )
    }


ds = ds.map(bicimlendir, remove_columns=ds.column_names)
print(ds[0]["text"][:600])

## 6) Eğit

In [ ]:
from trl import SFTConfig, SFTTrainer

trainer = SFTTrainer(
    model=model,
    train_dataset=ds,
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LENGTH,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,   # efektif batch = 8
        warmup_steps=5,
        num_train_epochs=3,
        learning_rate=2e-4,
        logging_steps=5,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=SEED,
        output_dir="outputs",
        report_to="none",
    ),
)

istatistik = trainer.train()
print(f"\nSüre: {istatistik.metrics['train_runtime'] / 60:.1f} dakika")
print(f"Son kayıp (loss): {istatistik.metrics['train_loss']:.4f}")

## 7) Eğitim sonrası cevaplar

Aynı beş soru. Farkı burada göreceğiz.

In [ ]:
FastLanguageModel.for_inference(model)

sonra = {}
for soru in TEST_SORULARI:
    sonra[soru] = sor(soru)
    print(f"\n{'=' * 70}\n### {soru}\n")
    print(f"--- ÖNCE ---\n{once[soru]}\n")
    print(f"--- SONRA ---\n{sonra[soru]}")

In [ ]:
# Karşılaştırmayı dosyaya yazıyorum ki repodaki rapora kopyalayabileyim.
import json

with open("karsilastirma.json", "w", encoding="utf-8") as fh:
    json.dump(
        [{"soru": s, "once": once[s], "sonra": sonra[s]} for s in TEST_SORULARI],
        fh, ensure_ascii=False, indent=2,
    )
print("karsilastirma.json yazıldı — indirip reports/ altına koy.")

## 8) Hugging Face'e yükle

**Token'ı buraya düz metin yazma.** Bu notebook GitHub'a commit'lenecek, token da onunla birlikte sızar.

Bunun yerine soldaki 🔑 **Secrets** panelinden `HF_TOKEN` adında bir gizli anahtar ekle (https://huggingface.co/settings/tokens adresinden **Write** yetkili token al) ve "Notebook access" anahtarını aç.

In [ ]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")
REPO = "berkcangumusisik/voleykoc-qwen3-4b-lora"

# Sadece LoRA adaptörü gidiyor (~100 MB), birleştirilmiş model değil (~8 GB).
# Ödev zaten "LoRA adaptörü yayımlanmalı" diyor.
model.push_to_hub(REPO, token=HF_TOKEN)
tokenizer.push_to_hub(REPO, token=HF_TOKEN)

print(f"Yüklendi -> https://huggingface.co/{REPO}")

## 9) Yüklenen adaptörü test et

Hub'dan geri yükleyip çalıştığını doğruluyorum — teslim etmeden önceki son kontrol.

In [ ]:
from unsloth import FastLanguageModel

test_model, test_tokenizer = FastLanguageModel.from_pretrained(
    model_name=REPO,          # adaptör; temel model otomatik indiriliyor
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(test_model)
print("Hub'dan yüklendi, adaptör çalışıyor.")